# Visual Sudoku Reasoning NCA
Neural Cellular Automata (NCA) solving visual Sudoku directly in 252×252 pixel space. Puzzles are sampled from the benchmark test set or supplied as custom boards. Clues are rendered using handwritten MNIST digits, and results are animated as inline rollout and prediction GIFs.
Please upload checkpoints/Visual-Sudoku/{test.csv, ema_params.npz} to the same folder as the notebook. We recommend Google Colab.

In [ ]:
#@title 1. Environment Setup
import base64
import io
import os
import urllib.request
import IPython.display as display
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

print(f"JAX Backend: {jax.default_backend().upper()} ({len(jax.devices())} device(s))")


In [ ]:
#@title 2. Load Model Weights, Dataset & MNIST
# 1. Load model weights from stripped checkpoint (layer0 & layer1 only)
ckpt_files = ["ema_params.npz", "checkpoints/Visual-Sudoku/ema_params.npz"]
ckpt_file = next((p for p in ckpt_files if os.path.exists(p)), None)
if not os.path.exists(ckpt_file):
  raise FileNotFoundError(f"Please provide '{ckpt_file}' in the working directory.")

ckpt = np.load(ckpt_file)
layer0 = jnp.array(ckpt["layer0"])
layer1 = jnp.array(ckpt["layer1"])

# 2. Load Sudoku-OOD Hard dataset
csv_paths = ["checkpoints/Visual-Sudoku/test.csv", "test.csv"]
csv_path = next((p for p in csv_paths if os.path.exists(p)), None)
if not csv_path:
  raise FileNotFoundError("Please provide 'test.csv' (or 'sudoku-hard/test.csv') in the working directory.")

with open(csv_path, "r") as f:
  puzzles = [line.strip().split(",") for line in f if line.strip()]
print(f"Loaded {len(puzzles)} puzzles from '{csv_path}'.")

# 3. Load MNIST digits (local file or public mirror fallback)
if os.path.exists("mnist.npz"):
  with np.load("mnist.npz") as f:
    x_test = f["x_test"].astype(np.float32) / 255.0
    y_test = f["y_test"]
    x_train = f["x_train"].astype(np.float32) / 255.0
    y_train = f["y_train"]
else:
  mnist_url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
  with np.load(io.BytesIO(urllib.request.urlopen(mnist_url).read())) as f:
    x_test = f["x_test"].astype(np.float32) / 255.0
    y_test = f["y_test"]
    x_train = f["x_train"].astype(np.float32) / 255.0
    y_train = f["y_train"]

# 4. Canonical digit prototypes computed from MNIST test set means (0=blank background, 1..9=class empirical test means)
canon = np.zeros((10, 28, 28), dtype=np.float32)
for d in range(1, 10):
  canon[d] = np.mean(x_test[y_test == d], axis=0)
canonical_digits = jnp.array(canon)

# Group MNIST train digits by class 1..9 for board clue synthesis
train_by_digit = {d: x_train[y_train == d] for d in range(1, 10)}
print(f"Loaded model weights ({layer0.shape}), dataset ({len(puzzles)} puzzles), prototypes from {len(x_test)} test digits, and {len(x_train)} train digits for board synthesis.")


In [ ]:
#@title 3. Select Puzzle & Generate Visual Sudoku
PUZZLE_INDEX = 9767 #@param {type:"integer"} # -1 for random puzzle, or 0..N-1 (e.g. 9767 is a solvable difficulty-56 puzzle)
CUSTOM_PUZZLE = "" #@param {type:"string"} # Optional: 81-character puzzle string
SEED = 42 #@param {type:"integer"} # -1 for random seed, or >= 0 for reproducible run

rng = np.random.default_rng(None if SEED < 0 else SEED)

# 1. Select puzzle
if CUSTOM_PUZZLE.strip():
  u_str = CUSTOM_PUZZLE.strip()
  s_str = None
  print("Using custom 81-digit puzzle input.")
elif 0 <= PUZZLE_INDEX < len(puzzles):
  u_str, s_str = puzzles[PUZZLE_INDEX][0], puzzles[PUZZLE_INDEX][1]
  print(f"Selected puzzle #{PUZZLE_INDEX} from test set.")
else:
  chosen_idx = int(rng.integers(len(puzzles)))
  u_str, s_str = puzzles[chosen_idx][0], puzzles[chosen_idx][1]
  print(f"Selected random puzzle #{chosen_idx} from test set.")

masked_board = np.array([int(c) for c in u_str], dtype=np.uint8).reshape(9, 9)
full_board = np.array([int(c) for c in s_str], dtype=np.uint8).reshape(9, 9) if s_str else None

# 2. Synthesize 252x252 visual puzzle: sample random MNIST train exemplars for clues, uniform noise for blanks
puzzle_img = rng.uniform(-0.1, 0.1, (252, 252)).astype(np.float32)
clue_count = 0
for r in range(9):
  for c in range(9):
    val = masked_board[r, c]
    if val > 0:
      exemplars = train_by_digit[val]
      puzzle_img[r * 28 : (r + 1) * 28, c * 28 : (c + 1) * 28] = exemplars[rng.integers(len(exemplars))]
      clue_count += 1

print(f"Generated 252x252 visual puzzle with {clue_count} clues (difficulty {81 - clue_count}).")


In [ ]:
#@title 4. Run Live NCA Simulation
STEPS = 3072 #@param {type:"integer"}
INTERVAL = 128 #@param {type:"integer"}

# Perception filters (Identity, Sobel-X, Sobel-Y)
ident = jnp.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=jnp.float32)
sobel_x = jnp.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=jnp.float32) / 8.0
sobel_y = jnp.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=jnp.float32) / 8.0
kernels = jnp.stack([ident, sobel_x, sobel_y])[:, None, :, :]

def nca_step(x, key):
  """Single stochastic NCA update step with periodic wrap boundary."""
  x_pad = jnp.pad(x, ((1, 1), (1, 1), (0, 0)), mode="wrap")
  x_t = jnp.transpose(x_pad, (2, 0, 1))[:, None, :, :]
  z = jax.lax.conv(x_t, kernels, (1, 1), "VALID")
  z = jnp.transpose(z, (2, 3, 1, 0)).reshape(256, 256, -1)
  dx = jax.nn.relu(z @ layer0) @ layer1
  fire_mask = (jr.uniform(key, dx.shape[:-1] + (1,)) <= 0.5).astype(jnp.float32)
  return x + dx * fire_mask

def interval_step(carry, _):
  """Sub-interval scan loop for JIT performance."""
  x, k = carry
  def step_fn(c, _):
    x_in, k_in = c
    k_in, subk = jr.split(k_in)
    return (nca_step(x_in, subk), k_in), None
  (x, k), _ = jax.lax.scan(step_fn, (x, k), None, length=INTERVAL)
  return (x, k), None

j_interval = jax.jit(interval_step)

@jax.jit
def classify_board(board_252):
  """Discretize 252x252 continuous state into 9x9 digits [0..9] via nearest prototype."""
  cells = board_252[:252, :252].reshape(9, 28, 9, 28).transpose(0, 2, 1, 3)
  l2 = jnp.sum(jnp.square(cells[:, :, None] - canonical_digits[None, None]), axis=(-1, -2))
  return jnp.argmin(l2, axis=-1)

def audit_board(grid, clues):
  """Check Sudoku rule validity and clue preservation."""
  expected = np.arange(1, 10)
  valid_rows = all(np.array_equal(np.sort(grid[r]), expected) for r in range(9))
  valid_cols = all(np.array_equal(np.sort(grid[:, c]), expected) for c in range(9))
  valid_boxes = all(
      np.array_equal(np.sort(grid[br * 3 : (br + 1) * 3, bc * 3 : (bc + 1) * 3].flatten()), expected)
      for br in range(3) for bc in range(3)
  )
  clue_mask = (clues > 0)
  clues_ok = np.all(grid[clue_mask] == clues[clue_mask])
  is_valid = valid_rows and valid_cols and valid_boxes and clues_ok
  preserved_count = int(np.sum(grid[clue_mask] == clues[clue_mask]))
  return {"is_valid": is_valid, "rows": valid_rows, "cols": valid_cols, "boxes": valid_boxes, "clues": preserved_count}

# Initialize simulation state: channel 0 has visual puzzle, channels 1..255 have noise
key = jr.PRNGKey(int(rng.integers(2**31)))
key, subk = jr.split(key)
x_state = rng.uniform(-0.1, 0.1, (256, 256, 256)).astype(np.float32)
x_state[..., 0] = np.pad(puzzle_img, [(0, 4), (0, 4)])

carry = (jnp.array(x_state), subk)
timesteps = [0]
visual_states = [np.asarray(carry[0][:252, :252, 0])]
predicted_grids = [np.asarray(classify_board(carry[0][:252, :252, 0]))]

print(f"Running NCA simulation ({STEPS} steps)...")
for step in range(INTERVAL, STEPS + 1, INTERVAL):
  carry, _ = j_interval(carry, None)
  v = np.asarray(carry[0][:252, :252, 0])
  pred = np.asarray(classify_board(v))

  timesteps.append(step)
  visual_states.append(v)
  predicted_grids.append(pred)

  audit = audit_board(pred, masked_board)
  if full_board is not None:
    matched = int(np.sum(pred == full_board))
    match_msg = f" | Matched: {matched:2d}/81 ({matched/81*100:5.1f}%)"
  else:
    match_msg = ""
  status = "[SOLVED]" if audit["is_valid"] else "Solving..."
  print(f"Step {step:4d}/{STEPS}{match_msg} | Clues: {audit['clues']}/{clue_count} | {status}")

audit_final = audit_board(predicted_grids[-1], masked_board)
print(f"\nFinal Audit: {audit_final}")


In [ ]:
#@title 5. Generate & Display Animated GIFs
def render_pred_frame(pred_grid, clues, full_solution, step):
  """Renders a clean 9x9 Sudoku frame with solid white background and centered title."""
  fig, ax = plt.subplots(figsize=(3.9, 3.9), dpi=75, facecolor="white")
  fig.patch.set_facecolor("white")
  ax.set_facecolor("white")
  ax.set_xlim(0, 9); ax.set_ylim(9, 0); ax.axis("off")

  # Grid lines (thick for 3x3 blocks, thin for cells)
  for i in range(10):
    ax.axhline(i, color="black" if i % 3 == 0 else "#9e9e9e", lw=2.0 if i % 3 == 0 else 0.8)
    ax.axvline(i, color="black" if i % 3 == 0 else "#9e9e9e", lw=2.0 if i % 3 == 0 else 0.8)

  for r in range(9):
    for c in range(9):
      val = int(pred_grid[r, c])
      if val > 0:
        is_clue = (clues[r, c] != 0)
        ax.text(c + 0.5, r + 0.5, str(val), ha="center", va="center",
                fontsize=12.5, fontweight="bold" if is_clue else "normal",
                color="#0d47a1" if is_clue else "#212121")

  # Format title cleanly on two lines to prevent overflowing the GIF width
  if full_solution is not None:
    matched = int(np.sum(pred_grid == full_solution))
    acc = matched / 81.0 * 100.0
    status = " [SOLVED]" if matched == 81 else ""
    title_str = f"Predictions (t={step})\nAccuracy: {matched}/81 ({acc:.0f}%){status}"
  else:
    audit = audit_board(pred_grid, clues)
    status = " [VALID]" if audit["is_valid"] else ""
    title_str = f"Predictions (t={step})\nClues: {audit['clues']}/{np.sum(clues > 0)}{status}"

  ax.set_title(title_str, fontsize=9.0, fontweight="bold", pad=5, color="#103463")
  fig.subplots_adjust(left=0.04, right=0.96, top=0.86, bottom=0.03)
  fig.canvas.draw()
  frame = np.asarray(fig.canvas.buffer_rgba())[..., :3]
  plt.close(fig)
  return Image.fromarray(frame)

# 1. Rollout GIF (Continuous Visual State on Black Background)
rollout_frames = [Image.fromarray((np.clip(v, 0.0, 1.0) * 255.0).astype(np.uint8)) for v in visual_states]
rollout_frames[0].save("rollout.gif", save_all=True, append_images=rollout_frames[1:], duration=150, loop=0)
print("Saved rollout.gif (black background)")

# 2. Predictions GIF on Solid White Background
pred_frames = [render_pred_frame(pred, masked_board, full_board, t) for pred, t in zip(predicted_grids, timesteps)]
pred_frames[0].save("predictions.gif", save_all=True, append_images=pred_frames[1:], duration=150, loop=0)
print("Saved predictions.gif (solid white background)")

# 3. Display GIFs inline in-cell via Base64 data URIs with explicit white container
def to_b64(path):
  with open(path, "rb") as f:
    return base64.b64encode(f.read()).decode("ascii")

b64_rollout = to_b64("rollout.gif")
b64_pred = to_b64("predictions.gif")

display.display(display.HTML(f"""
<div style="display: flex; flex-direction: row; gap: 24px; align-items: flex-start; margin-top: 10px; background: white; padding: 16px; border-radius: 8px; width: fit-content;">
  <div style="text-align: center; background: white;">
    <h4 style="margin: 0 0 8px 0; color: #103463; font-family: sans-serif;">Rollout (Visual State)</h4>
    <img src="data:image/gif;base64,{b64_rollout}" width="285" style="border: 1px solid #ddd; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.08);">
  </div>
  <div style="text-align: center; background: white;">
    <h4 style="margin: 0 0 8px 0; color: #103463; font-family: sans-serif;">Predictions (Discretized Grid & Accuracy)</h4>
    <img src="data:image/gif;base64,{b64_pred}" width="285" style="border: 1px solid #ddd; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.08);">
  </div>
</div>
"""))


In [ ]:
#@title 6. Generate 1x4 Publication Figure
# Renders the 1x4 horizontal figure (t=0, 128, 1024, 3072) with paper styling and colors.
target_ts = [0, 128, 1024, 3072]
avail_ts = [t for t in target_ts if t in timesteps]
if not avail_ts:
  indices = np.linspace(0, len(timesteps) - 1, 4, dtype=int)
  avail_ts = [timesteps[i] for i in indices]

fig, axes = plt.subplots(1, len(avail_ts), figsize=(3.5 * len(avail_ts), 3.8), dpi=300)
fig.subplots_adjust(wspace=0.04, left=0.01, right=0.99, top=0.90, bottom=0.02)
if len(avail_ts) == 1:
  axes = [axes]

for ax, t in zip(axes, avail_ts):
  idx = timesteps.index(t)
  ax.imshow(visual_states[idx][:252, :252], cmap="bone", vmin=0.0, vmax=1.0)
  ax.set_title(f"t={t}", fontsize=15, pad=6, color="#103463", fontfamily="sans-serif")
  ax.axis("off")

figure_path = "visual_sudoku_hard_paper_figure_1x4.png"
fig.savefig(figure_path, format="png", bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved publication figure: {figure_path}")
